# Design > Randomizer

<div class="alert alert-info">Randomly assign experimental units to treatment conditions</div>

The `randomizer` function performs random assignment of units (e.g., customers, stores, participants) to experimental conditions. It supports both simple complete randomization and block randomization (where balance is enforced within groups defined by a blocking variable).

In [1]:
import polars as pl
import pyrsm as rsm

In [2]:
## setup pyrsm for autoreload
%reload_ext autoreload
%autoreload 2
%aimport pyrsm

# Example: Simple random assignment

Suppose you have a list of 100 people and want to randomly assign them to two conditions: "test" and "control". Each person has an equal probability of being assigned to either condition.

In [3]:
rndnames = pl.read_parquet("../data/design/rndnames.parquet")
rndnames.head()

Names,Gender
str,enum
"""Ervin Escalona""","""Male"""
"""Allan Ammerman""","""Male"""
"""Milton Mothershed""","""Male"""
"""Deshawn Dawn""","""Male"""
"""Jc Julius""","""Male"""


In [4]:
r = rsm.design.randomizer(
    {"rndnames": rndnames},
    vars=["Names"],
    conditions=["test", "control"],
    seed=1234,
)
r.summary()

Random assignment (simple random)
Data         : rndnames
Variables    : Names
Conditions   : test, control
Probabilities: 0.5, 0.5
Random seed  : 1234
Duplicates   : Based on selected variables, no duplicate rows exist

Assignment frequencies:
  control: 50
  test: 50
  Total: 100

Assignment proportions:
  control: 0.5
  test: 0.5


With complete randomization and equal probabilities, each condition gets exactly 50 units. The `.conditions` column is added to the data showing the assignment:

In [5]:
r.data.head(10)

.conditions,Names
str,str
"""test""","""Ervin Escalona"""
"""control""","""Allan Ammerman"""
"""control""","""Milton Mothershed"""
"""control""","""Deshawn Dawn"""
"""control""","""Jc Julius"""
"""test""","""Denver Delph"""
"""control""","""Jed Jameson"""
"""control""","""Zachariah Zeno"""
"""control""","""Cyril Class"""


# Three or more conditions

You can assign units to any number of conditions. With 100 units and 3 conditions, each condition gets approximately 33-34 units:

In [6]:
r3 = rsm.design.randomizer(
    rndnames,
    vars=["Names"],
    conditions=["A", "B", "C"],
    seed=1234,
)
r3.summary()

Random assignment (simple random)
Data         : data
Variables    : Names
Conditions   : A, B, C
Probabilities: 0.333, 0.333, 0.333
Random seed  : 1234
Duplicates   : Based on selected variables, no duplicate rows exist

Assignment frequencies:
  A: 33
  B: 33
  C: 34
  Total: 100

Assignment proportions:
  A: 0.33
  B: 0.33
  C: 0.34


# Unequal probabilities

If you want to assign more units to one condition (e.g., 70% treatment, 30% control), use the `probs` parameter:

In [7]:
r_unequal = rsm.design.randomizer(
    rndnames,
    vars=["Names"],
    conditions=["treatment", "control"],
    probs=[0.7, 0.3],
    seed=1234,
)
r_unequal.summary()

Random assignment (simple random)
Data         : data
Variables    : Names
Conditions   : treatment, control
Probabilities: 0.7, 0.3
Random seed  : 1234
Duplicates   : Based on selected variables, no duplicate rows exist

Assignment frequencies:
  control: 30
  treatment: 70
  Total: 100

Assignment proportions:
  control: 0.3
  treatment: 0.7


# Block randomization

Block randomization ensures that within each level of a blocking variable, the assignment is balanced. This is useful when you want to ensure balance across important covariates (e.g., gender).

In this example, we block on `Gender` so that within both the Female and Male groups, half are assigned to "test" and half to "control":

In [8]:
r_block = rsm.design.randomizer(
    {"rndnames": rndnames},
    vars=["Names"],
    blocks="Gender",
    conditions=["test", "control"],
    seed=1234,
)
r_block.summary()

Random assignment (blocking)
Data         : rndnames
Variables    : Names
Blocks       : Gender
Conditions   : test, control
Probabilities: 0.5, 0.5
Random seed  : 1234
Duplicates   : Based on selected variables, no duplicate rows exist

Assignment frequencies:
shape: (2, 4)
┌────────┬─────────┬──────┬───────┐
│ Gender ┆ control ┆ test ┆ Total │
│ ---    ┆ ---     ┆ ---  ┆ ---   │
│ enum   ┆ u32     ┆ u32  ┆ u32   │
╞════════╪═════════╪══════╪═══════╡
│ Female ┆ 25      ┆ 25   ┆ 50    │
│ Male   ┆ 25      ┆ 25   ┆ 50    │
└────────┴─────────┴──────┴───────┘

Assignment proportions:
  control: 0.5
  test: 0.5


The cross-tabulation shows that within each gender group, the test and control conditions are perfectly balanced (25 each for both Female and Male).

© Vincent Nijs (2026)